# 05 — Mô hình 3: CNN (đóng băng) + GRU

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


In [ ]:
# === Thiết lập chung (Kaggle) ===
import sys, os
# Cách 1: clone repo (khuyến nghị để đồng bộ code):
# !git clone https://github.com/ducqwang/FallDetection.git /kaggle/working/FD
# REPO='/kaggle/working/FD'
# Cách 2: notebook nằm sẵn trong repo:
REPO = '..' if os.path.exists('../src') else '/kaggle/working/FD'
sys.path.insert(0, REPO)
from src.utils import set_seed, load_config, on_kaggle
cfg = load_config(os.path.join(REPO, 'configs/default.yaml'))
set_seed(cfg['seed'])
# TRÊN KAGGLE: sửa cfg['paths'] trỏ tới /kaggle/input/<dataset-slug>/...
# cfg['paths']['urfd_root'] = '/kaggle/input/urfd/...'
# cfg['paths']['le2i_root'] = '/kaggle/input/le2i/...'
print('Kaggle:', on_kaggle())


In [ ]:
!pip -q install ultralytics lap


In [ ]:
from src.detection_tracking import DetectorTracker
from src.data_splits import index_le2i
from src.pipeline import build_crop_embedding_samples
from src.datasets import CropSequenceDataset, pad_collate
from src.models.model3_cnn_gru import SeqGRU, train
from torch.utils.data import DataLoader
dt = DetectorTracker(cfg['detection']['model'])
items = index_le2i(cfg['paths']['le2i_root'])
train_items, val_items = items[:120], items[120:]
# Tiền tính embedding CNN (backbone đóng băng) -> nhanh hội tụ, nhẹ GPU
tr = build_crop_embedding_samples(train_items, dt, cfg['model3_cnn_gru']['backbone'], cfg['data']['seq_len'])
va = build_crop_embedding_samples(val_items, dt, cfg['model3_cnn_gru']['backbone'], cfg['data']['seq_len'])
ltr = DataLoader(CropSequenceDataset(tr), batch_size=32, shuffle=True, collate_fn=pad_collate)
lva = DataLoader(CropSequenceDataset(va), batch_size=32, collate_fn=pad_collate)
D = tr[0][0].shape[-1]
model = SeqGRU(in_dim=D, hidden=cfg['model3_cnn_gru']['hidden'])
train(model, ltr, lva, cfg['model3_cnn_gru'], 'outputs/model3_best.pt')
